# 🔍 GIADA Task 9b — origine del reference floor
Rianalisi forense degli stessi 350 percorsi Task 9. Nessun nuovo teacher, nessuna rete o GPU necessaria. Confrontiamo solo sei integrazioni preregistrate della formula Ca_HVA.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_9b');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not GIADA_REPO.exists() and not TEACHER_REPO.exists(),'Sessione già inizializzata: usa una sessione Kaggle nuova.'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula,run_physiological_path_floor
from src.giada_teacher.physiological_path_floor import load_verified_task9,EXPECTED_TASK9_ZIP_SHA256
plan=json.loads((GIADA_REPO/'experiments/teacher_physiological_path_floor_plan_v1.json').read_text())
display({'task':plan['task'],'methods':list(plan['methods']),'study_type':plan['study_type'],'gpu_required':False})


## 🔎 Input necessari
Aggiungi agli input Kaggle il dataset targeted v1.1 base e lo ZIP `giada_physiological_voltage_paths.zip` della Task 9. Quest'ultimo è piccolo; non serve più l'artefatto Task 5.

In [ ]:
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK9_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_physiological_voltage_paths.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'physiological' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('selected_paths.json') if (p.parent/'final_report.json').is_file()]
TASK9_SOURCE=None
for path in candidates:
 if not path.exists():continue
 try:load_verified_task9(path);TASK9_SOURCE=path.resolve();break
 except (RuntimeError,FileNotFoundError,ValueError,KeyError):continue
assert TASK9_SOURCE is not None,'ZIP Task 9 esatto non trovato. Aggiungi giada_physiological_voltage_paths.zip o imposta GIADA_TASK9_ARTIFACT.'
base_override=os.environ.get('GIADA_TARGETED_DATASET')
base_candidates=[Path(base_override).expanduser()] if base_override else []
base_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-targeted-transition-dataset-v1-1-base')]
if INPUT_ROOT.is_dir():base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
BASE_ROOT=next((p.resolve() for p in base_candidates if p.is_dir() and all((p/name).is_file() for name in ('transition_dataset.h5','state_schema.json','dataset_manifest.json'))),None)
assert BASE_ROOT is not None,'Dataset targeted v1.1 base estratto non trovato. Aggiungilo agli Input oppure imposta GIADA_TARGETED_DATASET alla cartella con HDF5/manifest/schema.'
print({'task9_source':str(TASK9_SOURCE),'base_root':str(BASE_ROOT),'task9_hash_verified':True})


In [ ]:
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_physiological_path_floor_forensic')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Usa una sessione nuova.'
def progress(percent,label):print(f'[GIADA Task 9b][SHA-256 {label}] {percent}%',flush=True)
report=run_physiological_path_floor(formula,BASE_ROOT,TASK9_SOURCE,OUTPUT_DIR,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'paths':report['path_count'],'task9_reproduction_error':report['task9_right_floor_reproduction_max_error'],'constant_voltage_refinement_error':report['right_substep_invariance_max_abs'],'spike_interpolation_gate':report['all_spike_groups_meet_preregistered_interpolation_gate'],'new_teacher_paths':report['new_teacher_paths_generated']})
assert report['valid'] and report['task9_right_floor_reproduction_max_error']<=1e-10 and report['right_substep_invariance_max_abs']<=1e-10


In [ ]:
import pandas as pd
rows=[]
for key,value in report['groups'].items():
 m=value['methods']
 rows.append({'site_regime':key,'n':value['count'],'right_m':round(m['right']['m_rmse'],6),'left_m':round(m['left']['m_rmse'],6),'mid_m':round(m['midpoint']['m_rmse'],6),'linear5_m':round(m['linear_5']['m_rmse'],6),'linear20_m':round(m['linear_20']['m_rmse'],6),'fraction_improved':round(value['linear20_fraction_paths_m_improved_vs_right'],3)})
display(pd.DataFrame(rows))
if not report['all_spike_groups_meet_preregistered_interpolation_gate']:print('Il reference floor negli spike resta irrisolto secondo la soglia preregistrata; serve una piccola registrazione teacher più fitta, non una conclusione sulla LUT.')


## 📦 Scarica lo ZIP con il metodo Blob/base64
Lo ZIP contiene il report piccolo, non il dataset HDF5.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_physiological_path_floor_forensic','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
